# Dense Vector Embeddings (Sentence-Transformers)

Upgrades the TF-IDF baseline to semantic matching: instead of exact word overlap,
resumes and job descriptions are embedded into dense vectors that capture *meaning*,
so "Java Spring Boot engineer" can match a job description about "software engineering"
even with zero shared words — the exact gap the TF-IDF baseline notebook flagged.

Steps:
1. Load the pretrained `all-MiniLM-L6-v2` Sentence-Transformers model
2. Embed all resumes and all job descriptions
3. Score every resume against every job description with cosine similarity
4. Compare top matches against the TF-IDF baseline

Assumes `preprocess.ipynb` has already been run (so `data/processed/resumes_clean.csv`
and `data/processed/job_descriptions.csv` exist). First run downloads the model
(~80MB) and requires internet access.


In [ ]:
%run config.ipynb

## 1. Load cleaned data

In [ ]:
import pandas as pd

resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
job_descriptions = pd.read_csv(JOB_DESCRIPTIONS_CSV)

print(f"Loaded {len(resumes_clean)} resumes and {len(job_descriptions)} job descriptions")

## 2. Load the embedding model
`all-MiniLM-L6-v2` is a good default: small (~80MB), fast on CPU, and strong general-purpose
semantic performance — a common choice for this kind of similarity task.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print(f"Loaded model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 3. Embed resumes and job descriptions

In [ ]:
resume_embeddings = model.encode(
    resumes_clean["clean_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,  # so cosine similarity == dot product
)

job_embeddings = model.encode(
    job_descriptions["job_description"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Resume embeddings shape:", resume_embeddings.shape)
print("Job description embeddings shape:", job_embeddings.shape)

## 4. Score resumes against a job description

In [ ]:
import numpy as np

def top_matches_for_category(category: str, top_k: int = 5) -> pd.DataFrame:
    """Return the top_k resumes ranked by embedding cosine similarity to the
    given job description category. Embeddings are normalized, so a simple
    dot product IS the cosine similarity."""
    job_idx = job_descriptions.index[job_descriptions[COL_CATEGORY] == category]
    if len(job_idx) == 0:
        raise ValueError(f"No job description found for category: {category}")
    job_idx = job_idx[0]

    sims = resume_embeddings @ job_embeddings[job_idx]
    results = resumes_clean.copy()
    results["similarity"] = sims
    return results.sort_values("similarity", ascending=False).head(top_k)[
        [COL_ID, COL_CATEGORY, "similarity"]
    ]

example_category = job_descriptions[COL_CATEGORY].iloc[0]
print(f"Top matches for category: {example_category}\n")
top_matches_for_category(example_category)

## 5. Compare against the TF-IDF baseline
Loads the saved TF-IDF artifacts (from `tfidf_baseline.ipynb`) and ranks the same
category with both methods side by side, so you can see where semantic matching
picks up resumes that exact-word matching missed.

In [ ]:
import joblib

TFIDF_DIR = PROCESSED_DATA_DIR / "tfidf"

try:
    tfidf_vectorizer = joblib.load(TFIDF_DIR / "tfidf_vectorizer.joblib")
    tfidf_resume_vectors = joblib.load(TFIDF_DIR / "resume_vectors.joblib")
    tfidf_job_vectors = joblib.load(TFIDF_DIR / "job_vectors.joblib")

    from sklearn.metrics.pairwise import cosine_similarity

    job_idx = job_descriptions.index[job_descriptions[COL_CATEGORY] == example_category][0]
    tfidf_sims = cosine_similarity(tfidf_job_vectors[job_idx], tfidf_resume_vectors).flatten()

    comparison = resumes_clean[[COL_ID, COL_CATEGORY]].copy()
    comparison["tfidf_similarity"] = tfidf_sims
    comparison["embedding_similarity"] = resume_embeddings @ job_embeddings[job_idx]

    display(comparison.sort_values("embedding_similarity", ascending=False))
except FileNotFoundError:
    print(
        "TF-IDF artifacts not found — run tfidf_baseline.ipynb first "
        "if you want the side-by-side comparison."
    )

## 6. Save embeddings for reuse

In [ ]:
EMBEDDINGS_DIR = PROCESSED_DATA_DIR / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

np.save(EMBEDDINGS_DIR / "resume_embeddings.npy", resume_embeddings)
np.save(EMBEDDINGS_DIR / "job_embeddings.npy", job_embeddings)
resumes_clean[[COL_ID, COL_CATEGORY]].to_csv(EMBEDDINGS_DIR / "resume_index.csv", index=False)
job_descriptions[[COL_CATEGORY]].to_csv(EMBEDDINGS_DIR / "job_index.csv", index=False)

print(f"Saved embeddings to {EMBEDDINGS_DIR}")

---
## Notes
- Embeddings are **normalized** (`normalize_embeddings=True`), so cosine similarity
  reduces to a dot product — faster to compute at scale.
- `all-MiniLM-L6-v2` is a solid default; if resumes are highly technical/domain-specific,
  a larger model (`all-mpnet-base-v2`) may improve accuracy at the cost of speed.
- The `*_index.csv` files preserve the row order that lines up with the saved `.npy`
  embedding arrays — needed if you reload them later without recomputing.
